# Image informativeness in disaster posts
**Khush Bhavin Maru | COMP6001 Computer Vision and Multimodal Machine Learning**

This portfolio edition reviews the measured capstone experiment and runs the source checks that need no dataset. The classifier predicts the informativeness of an image from a disaster-related post, with optional accompanying text. Run the notebook from the repository root after installing the requirements.

In [ ]:
from pathlib import Path
import subprocess
import sys

if not Path('src/models.py').is_file():
    raise FileNotFoundError('Start Jupyter from the repository root.')

## 1. Fixed experiment and dataset audit

`protocol.json` records the model list, seeds and selection rules fixed before test evaluation. The split groups together repeated tweet IDs, identical decoded image pixels and sufficiently long identical cleaned texts. The text vocabulary and SVD are fitted on training data only. The visual encoder is pretrained and frozen.

In [ ]:
import json
import numpy as np
import pandas as pd
from IPython.display import display, Image

protocol = json.loads(Path('protocol.json').read_text())
audit = json.loads(Path('results/data_audit.json').read_text())
print('Dataset:', protocol['dataset'], '| fixed revision:', protocol['revision'])
print('Records:', audit['rows'], '| groups:', audit['groups'])
display(pd.DataFrame(audit['splits']))
print('Original split shared groups:', audit['original_split_group_overlap'])

In [ ]:
display(Image(filename='results/figures/dataset_distributions.png'))

## 2. Model implementation

Text uses training-fitted TF-IDF and a 256-dimensional SVD. The image branch uses 512-dimensional frozen ResNet18 features. Each learned projection has 128 hidden units. The gate combines these features with availability flags, their masked absolute difference and element-wise product. Auxiliary heads provide a direct supervised signal to each branch.

The implementation below is the actual source used in training. PyTorch supplies tensor operations and optimisation; the fusion network and training loop are written directly. The pretrained encoder is credited in README.md. Gating and modality dropout build on established methods, cited in the report.

In [ ]:
import inspect
from src.models import Fusion
from src.train import run as train_models
print(inspect.getsource(Fusion))
model = Fusion('gated')
print('Custom model trainable parameters:', sum(p.numel() for p in model.parameters()))

## 3. Checks

The model masks an unavailable modality after its projection, including the bias. The invariance check changes the unavailable raw input and requires the logits to agree. This checkout runs the architecture, gradient, configuration and cache-identity checks. The verifier also checks the split and saved metrics inside a reproduced run.

In [ ]:
from src.verify import run as verify
verify()

## 4. Benchmarks and clean-test measurements

The text and image MLPs test each source separately. Concatenation learns a joint representation; late fusion averages unimodal decisions. Concatenation with modality dropout is the prespecified matched control. Each model uses seeds 42, 43 and 44. The test partition is never used to choose an epoch.

The table reads aggregate metrics from the submitted run. Seed SD describes optimisation variation; it is not a population confidence interval.

In [ ]:
metrics = pd.read_csv('results/metrics.csv')
clean = metrics.query("condition == 'clean'")
display(clean.groupby('model')[['accuracy','macro_f1','macro_precision','macro_recall','auroc']].agg(['mean','std']).round(4))
display(Image(filename='results/figures/comparison_ablations.png'))

## 5. Learning curves and error analysis

Checkpoints use validation macro-F1, with earliest tie-breaking. Test curves are retrospective evaluations of saved epoch states, computed only after the checkpoint choices were locked. They illustrate generalisation but must not be used to retune this experiment.

In [ ]:
display(Image(filename='results/figures/learning_curves.png'))
display(Image(filename='results/figures/classification_diagnostics.png'))

## 6. Controlled missing and mismatched inputs

Text removal and image removal use explicit availability flags. Text shuffling preserves each image and its target, then substitutes another held-out text using a fixed permutation. This breaks the pairing without changing the visual ground truth. These are synthetic stress tests, not estimated real outage frequencies.

In [ ]:
display(Image(filename='results/figures/robustness.png'))
display(Image(filename='results/figures/events_gates.png'))

## 7. Group-aware statistical comparison

The primary comparison is custom fusion versus concatenation with modality dropout on clean-test macro-F1. Predictions are averaged across seeds. Bootstrap intervals and paired randomisation tests operate on duplicate groups, so images within one group are not treated as independent samples. Secondary robustness p-values receive Holm correction. The tests are conditional on this split and these fitted models.

In [ ]:
statistics = json.loads(Path('results/statistics.json').read_text())
stat_rows = []
for condition, result in statistics.items():
    stat_rows.append({'condition': condition, 'ensemble_F1_difference': result['difference_macro_f1'],
                      'bootstrap_CI95': result['bootstrap_ci95'], 'paired_p': result['permutation_p_two_sided'],
                      'secondary_Holm_p': result.get('secondary_holm_p', np.nan)})
display(pd.DataFrame(stat_rows))
display(Image(filename='results/figures/feature_distributions.png'))

## 8. Full reproduction

Set `RUN_FULL_REPRODUCTION = True` to repeat the experiment. The command creates a fresh folder under `runs/` and leaves the submitted report, model weights and results intact. It downloads the pinned source data and rebuilds the split, features, models, evaluations and figures. No manual deletion is required.

Visual extraction supports a Colab CUDA GPU or local MPS GPU. Cached fusion heads train on CPU. If this project already has a complete `data/raw/` folder, set `REUSE_RAW_DOWNLOADS=True` to reuse those raw files while still rebuilding features and models. See README.md for stage-by-stage resumption after an interruption.

In [ ]:
RUN_FULL_REPRODUCTION = False
REUSE_RAW_DOWNLOADS = False
if RUN_FULL_REPRODUCTION:
    command = [sys.executable, '-m', 'src', 'reproduce']
    if REUSE_RAW_DOWNLOADS:
        command.append('--reuse-data')
    subprocess.run(command, check=True)
else:
    print('Reviewing the included measured run. Full reproduction writes to a separate fresh folder.')

## 9. Demonstration

After full reproduction, run the prediction command in the new experiment directory. It uses the rebuilt text pipeline and the three selected gated checkpoints. Provide your own image and optionally supply text.

In [ ]:
print('python -m src.predict --image /path/to/image.jpg --text "Flood water covers the road"')

## Limits and interpretation

This is a historical evaluation on one grouped CrisisMMD split. Event coverage, annotation disagreement and frozen features limit generalisation. Exact duplicate matching can miss similar crops or edits. The classifier is an academic experiment. See the README for sources and evaluation details.